In [0]:
%sql
-- Standard calendar (date dimension) table: one row per day from 1940-01-01 to 2030-12-31
CREATE OR REPLACE TABLE adb_healthcare.silver.calendar
COMMENT 'Standard calendar dimension, one row per day from 1940-01-01 to 2030-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS date
)
SELECT
  CAST(date_format(date, 'yyyyMMdd') AS INT) AS date_key,      -- e.g. 20260131 (join key used by gold, see STTM)
  date                                       AS date,
  day(date)                                  AS day,
  date_format(date, 'EEEE')                  AS day_name,      -- Monday
  month(date)                                AS month,
  date_format(date, 'MMMM')                  AS month_name,    -- January
  quarter(date)                              AS quarter,
  concat('Q', quarter(date))                 AS quarter_name,  -- Q1..Q4
  year(date)                                 AS year,
  dayofweek(date) IN (1, 7)                  AS is_weekend     -- TRUE for Saturday/Sunday
FROM dates
ORDER BY date;

In [0]:
%sql
-- Sample rows for well-known dates (boundaries, leap day, year edges, today)
SELECT *
FROM adb_healthcare.silver.calendar
WHERE date IN (
  DATE'1940-01-01',  -- first date (Monday)
  DATE'1969-07-20',  -- Sunday
  DATE'2000-02-29',  -- leap day (century leap year)
  DATE'2021-01-01',  -- Friday
  DATE'2024-12-31',  -- Tuesday
  DATE'2026-10-27',  -- Tuesday (today)
  DATE'2030-12-31'   -- last date
)
ORDER BY date;

In [0]:
%sql
-- Each test returns expected vs actual and PASS/FAIL
WITH c AS (SELECT * FROM adb_healthcare.silver.calendar),
tests AS (
  SELECT 'Row count = days between 1940-01-01 and 2030-12-31' AS test,
         CAST(datediff(DATE'2030-12-31', DATE'1940-01-01') + 1 AS STRING) AS expected,
         CAST((SELECT count(*) FROM c) AS STRING) AS actual
  UNION ALL SELECT 'Min date', '1940-01-01', CAST((SELECT min(date) FROM c) AS STRING)
  UNION ALL SELECT 'Max date', '2030-12-31', CAST((SELECT max(date) FROM c) AS STRING)
  UNION ALL SELECT 'date_key is unique', CAST((SELECT count(*) FROM c) AS STRING), CAST((SELECT count(DISTINCT date_key) FROM c) AS STRING)
  UNION ALL SELECT 'No gaps between consecutive dates', '0',
         CAST((SELECT count(*) FROM (SELECT datediff(date, lag(date) OVER (ORDER BY date)) AS d FROM c) WHERE d <> 1) AS STRING)
  UNION ALL SELECT 'No NULLs in key columns', '0',
         CAST((SELECT count(*) FROM c WHERE date_key IS NULL OR date IS NULL OR is_weekend IS NULL OR year IS NULL OR month IS NULL OR quarter IS NULL OR day_name IS NULL) AS STRING)
  UNION ALL SELECT 'Number of years (1940-2030)', '91', CAST((SELECT count(DISTINCT year) FROM c) AS STRING)
  UNION ALL SELECT 'Every year has 365 or 366 days', '0',
         CAST((SELECT count(*) FROM (SELECT year, count(*) n FROM c GROUP BY year) WHERE n NOT IN (365, 366)) AS STRING)
  UNION ALL SELECT 'Leap days (Feb 29) count', '23',
         CAST((SELECT count(*) FROM c WHERE month = 2 AND day = 29) AS STRING)
  UNION ALL SELECT 'Quarter consistent with month', '0',
         CAST((SELECT count(*) FROM c WHERE quarter <> CAST(ceil(month / 3.0) AS INT)) AS STRING)
  UNION ALL SELECT 'quarter_name = Q + quarter', '0',
         CAST((SELECT count(*) FROM c WHERE quarter_name <> concat('Q', quarter)) AS STRING)
  UNION ALL SELECT 'date_key = yyyyMMdd of date', '0',
         CAST((SELECT count(*) FROM c WHERE date_key <> year * 10000 + month * 100 + day) AS STRING)
  UNION ALL SELECT 'is_weekend TRUE only for Saturday/Sunday', '0',
         CAST((SELECT count(*) FROM c WHERE is_weekend <> (day_name IN ('Saturday', 'Sunday'))) AS STRING)
  UNION ALL SELECT '2026-10-24 (Saturday) is weekend, 2026-10-27 (Tuesday) is not', 'true|false',
         (SELECT concat_ws('|', max(CASE WHEN date = DATE'2026-10-24' THEN CAST(is_weekend AS STRING) END), max(CASE WHEN date = DATE'2026-10-27' THEN CAST(is_weekend AS STRING) END)) FROM c)
  UNION ALL SELECT '12 distinct month names', '12', CAST((SELECT count(DISTINCT month_name) FROM c) AS STRING)
  UNION ALL SELECT '7 distinct day names', '7', CAST((SELECT count(DISTINCT day_name) FROM c) AS STRING)
  UNION ALL SELECT '1940-01-01 is Monday', 'Monday', (SELECT day_name FROM c WHERE date = DATE'1940-01-01')
  UNION ALL SELECT '2026-10-27 is Tuesday in October, Q4', 'Tuesday|October|Q4', (SELECT concat_ws('|', day_name, month_name, quarter_name) FROM c WHERE date = DATE'2026-10-27')
)
SELECT test, expected, actual, CASE WHEN expected = actual THEN 'PASS' ELSE 'FAIL' END AS result
FROM tests;